In [1]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls

Mounted at /content/drive
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [ ]:
!pip install -e .

In [1]:
import slicegpt
from slicegpt import rotate, model_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)

slicegpt imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/model_utils.py


In [2]:
import subprocess
import os
from datetime import datetime

# -----------------------------
# Config
# -----------------------------
LOG_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_projection"
MODEL_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

CAL_DATASETS = ["wikitext2", "squad2"]

# Define the comparison modes you want
PROJECTION_MODES = {
    "attn_only": {"project_attention": True,  "project_mlp": False},
    "mlp_only":  {"project_attention": False, "project_mlp": True},
     "both":     {"project_attention": True,  "project_mlp": True},   # optional
}


def run_slicegpt_projection_only(dataset: str, sparsity: float, mode: str) -> None:
    """
    Projection-only run:
    - attn_only: project attention submodules only
    - mlp_only:  project MLP submodules only
    """

    if mode not in PROJECTION_MODES:
        raise ValueError(f"Unknown mode={mode}. Choose from: {list(PROJECTION_MODES.keys())}")

    pa = PROJECTION_MODES[mode]["project_attention"]
    pm = PROJECTION_MODES[mode]["project_mlp"]

    run_tag = f"{dataset}_s{sparsity:.2f}_projOnly_{mode}".replace(".", "p")
    log_path = os.path.join(LOG_DIR, run_tag + ".txt")
    save_dir = os.path.join(MODEL_DIR, run_tag)
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python", "-B",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", "Qwen/Qwen2-0.5B",
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", "8",
        "--save-rotation-matrices",

        # ---- PROJECTION ONLY ----
        "--projection-only",
    ]

    # Toggle projectors (run_slicegpt enforces: projection-only requires at least one enabled) :contentReference[oaicite:1]{index=1}
    if not pa:
        cmd.append("--no-project-attention")
    if not pm:
        cmd.append("--no-project-mlp")

    print("\n=====================================================")
    print(f"Running PROJECTION-ONLY | dataset={dataset}, sparsity={sparsity}, mode={mode}")
    print(f"  project_attention={pa}, project_mlp={pm}")
    print("Command:", " ".join(cmd))
    print("Log file:", log_path)
    print("Save dir:", save_dir)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            f.write(line)

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())


# -----------------------------
# Example sweep
# -----------------------------
# for ds in CAL_DATASETS:
#     for mode in ["attn_only", "mlp_only"]:
#         run_slicegpt_projection_only(ds, sparsity=0.25, mode=mode)


In [ ]:
for ds in CAL_DATASETS:
     for mode in PROJECTION_MODES:
         run_slicegpt_projection_only(ds, sparsity=0.10, mode=mode)

In [3]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
!pip uninstall -y lm-eval lm_eval
!pip install "lm-eval==0.4.2"

In [ ]:
!pip uninstall -y peft
!pip install "peft==0.10.0"

In [22]:
# =========================
# CELL 1 — Eval function for PROJECTION-ONLY via ROTATION MATRICES (hooks)
# =========================
import os, json, logging
import torch
import lm_eval
from lm_eval import utils as lm_eval_utils
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager

from slicegpt import hf_utils
from slicegpt.config import config

logger = logging.getLogger(__name__)

def _as_tensor(x):
    return x if isinstance(x, torch.Tensor) else torch.tensor(x)

def _iter_entries(rotation_matrices):
    if isinstance(rotation_matrices, dict):
        return rotation_matrices.values()
    if isinstance(rotation_matrices, list):
        return rotation_matrices
    raise TypeError(f"rotation_matrices must be list or dict, got {type(rotation_matrices)}")

def _get_projector(entry, device, dtype):
    # prefer stored projector; fallback to eigenvectors->P
    if isinstance(entry, dict) and entry.get("projector") is not None:
        P = _as_tensor(entry["projector"])
    else:
        Q = None
        for k in ("eigenvectors", "evecs", "U", "rotation_matrix", "R"):
            if isinstance(entry, dict) and entry.get(k) is not None:
                Q = _as_tensor(entry[k])
                break
        if Q is None:
            raise ValueError(f"Entry missing projector/eigenvectors. Keys={list(entry.keys()) if isinstance(entry, dict) else type(entry)}")
        P = Q @ Q.T
    return P.to(device=device, dtype=dtype)

def apply_projection_hooks(model_adapter, rotation_matrices, mode: str):
    """
    Attach projection-only hooks by directly accessing HF modules (robust path finding).

    rotation entry schema (yours):
      - entry["type"]: "attention_projector" or "mlp_projector"
      - entry["layer"]: int
      - entry["projector"]: [d,d]
      - entry["enabled"]: bool
    """
    model = model_adapter.model  # e.g., Qwen2ForCausalLM
    device = model.device
    dtype  = next(model.parameters()).dtype
    hooks = []

    # ---- find transformer block list robustly ----
    layers = None
    candidate_paths = [
        "model.layers",            # common for HF *ForCausalLM wrappers (LLaMA/Qwen2)
        "layers",                  # some base models
        "transformer.h",           # GPT-style
        "model.decoder.layers",    # some encoder-decoder/decoder wrappers
        "decoder.layers",          # some decoders
    ]
    for path in candidate_paths:
        obj = model
        ok = True
        for attr in path.split("."):
            if not hasattr(obj, attr):
                ok = False
                break
            obj = getattr(obj, attr)
        if ok:
            layers = obj
            break

    if layers is None:
        raise AttributeError(
            "Could not locate transformer layers. Tried: "
            + ", ".join(candidate_paths)
            + f". Top-level attrs: {sorted([a for a in dir(model) if not a.startswith('_')])[:50]} ..."
        )

    # ensure indexable
    try:
        n_layers = len(layers)
    except Exception:
        raise TypeError(f"Found layers at '{path}' but it is not list-like: type={type(layers)}")

    matched_attn = 0
    matched_mlp  = 0

    entries = rotation_matrices if isinstance(rotation_matrices, list) else list(rotation_matrices.values())

    for entry in entries:
        if not isinstance(entry, dict):
            continue
        if not entry.get("enabled", True):
            continue

        layer = int(entry["layer"])
        if layer < 0 or layer >= n_layers:
            continue

        t = str(entry.get("type", "")).lower()
        is_attn = "attention" in t      # "attention_projector"
        is_mlp  = "mlp" in t            # "mlp_projector"

        # projector
        P = entry.get("projector", None)
        if P is None:
            Q = entry.get("eigenvectors", None)
            if Q is None:
                continue
            Q = Q if isinstance(Q, torch.Tensor) else torch.tensor(Q)
            P = Q @ Q.T
        P = P.to(device=device, dtype=dtype)

        def pre_hook(module, inputs, P=P):
            x = inputs[0]
            return (x @ P.T,) + inputs[1:]

        block = layers[layer]

        if is_attn and mode in ("attn_only", "both"):
            # Qwen2/LLaMA-like naming
            attn = getattr(block, "self_attn", None) or getattr(block, "attention", None)
            if attn is None:
                raise AttributeError(f"Layer {layer} has no self_attn/attention module. attrs={dir(block)}")

            # Try common projection names
            q = getattr(attn, "q_proj", None) or getattr(attn, "q_proj", None)
            k = getattr(attn, "k_proj", None)
            v = getattr(attn, "v_proj", None)

            if q is None or k is None or v is None:
                raise AttributeError(f"Attention layer {layer} missing q/k/v proj. attn attrs={dir(attn)}")

            hooks += [
                q.register_forward_pre_hook(pre_hook),
                k.register_forward_pre_hook(pre_hook),
                v.register_forward_pre_hook(pre_hook),
            ]
            matched_attn += 1

        if is_mlp and mode in ("mlp_only", "both"):
            mlp = getattr(block, "mlp", None) or getattr(block, "feed_forward", None)
            if mlp is None:
                raise AttributeError(f"Layer {layer} has no mlp/feed_forward module. attrs={dir(block)}")

            up   = getattr(mlp, "up_proj", None)
            gate = getattr(mlp, "gate_proj", None)

            if up is None or gate is None:
                raise AttributeError(f"MLP layer {layer} missing up/gate proj. mlp attrs={dir(mlp)}")

            hooks += [
                up.register_forward_pre_hook(pre_hook),
                gate.register_forward_pre_hook(pre_hook),
            ]
            matched_mlp += 1

    print(f"[HOOKS] layer_path='{path}' n_layers={n_layers} | hooks={len(hooks)} | matched_attn={matched_attn} | matched_mlp={matched_mlp}")
    return hooks


def eval_projection_only_from_rot(args):
    """
    Projection-only eval (correct for your current implementation):
    - Load base HF model (full width)
    - Load rotation matrices (projectors)
    - Register projection hooks according to mode
    - Run lm-eval
    """
    logger.info("Running Evaluation (projection-only via rotation matrices/hooks)")
    logger.info(f"Base model: {args['model']}")
    logger.info(f"Rotation matrices: {args['rot_path']}")
    logger.info(f"Mode: {args['mode']} | Sparsity tag: {args['sparsity']:.2f}")

    # base model
    model_adapter, tokenizer = hf_utils.get_model_and_tokenizer(
        args["model"], model_path=None, token=None, dtype=args.get("dtype", torch.float16)
    )
    model_adapter.model.to(config.device)
    model_adapter.model.eval()

    # pad token safety
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # apply hooks
    rot = torch.load(args["rot_path"], map_location="cpu")
    hooks = apply_projection_hooks(model_adapter, rot, mode=args["mode"])
    print("hooks:", len(hooks))

    # sanity: check first entry projector isn't identity-ish
    rot = torch.load(args["rot_path"], map_location="cpu")
    entry0 = next(iter(rot)) if isinstance(rot, list) else next(iter(rot.values()))
    P = entry0.get("projector", None)
    if P is not None:
        P = P.float()
        print("P diag mean:", P.diag().mean().item(), "P diag std:", P.diag().std().item())
        print("P offdiag mean abs:", (P - torch.diag(P.diag())).abs().mean().item())



    logger.info(f"Attached hooks: {len(hooks)}")

    # lm-eval wrapper
    hflm = HFLM(
        pretrained=model_adapter.model,
        tokenizer=tokenizer,
        batch_size=args["batch_size"],
    )

    # tasks
    tm = args.get("task_manager") or TaskManager()
    all_task_names = list(tm.all_tasks) if isinstance(tm.all_tasks, list) else list(tm.all_tasks.keys())
    patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
    task_names = lm_eval_utils.pattern_match(patterns, all_task_names)
    logger.info(f"Selected Tasks: {task_names}")

    # run
    results = lm_eval.simple_evaluate(
        model=hflm,
        tasks=task_names,
        task_manager=tm,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=False,
        log_samples=False,
    )["results"]

    # save
    os.makedirs(args["save_dir"], exist_ok=True)
    out_path = os.path.join(
        args["save_dir"],
        f"results_projOnlyHooks_s{args['sparsity']:.2f}_{args['mode']}_{'_'.join(task_names)}.json",
    )
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {out_path}")
    return results


In [16]:
import torch

rot = torch.load("/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt", map_location="cpu")

print(type(rot), "len:", len(rot) if isinstance(rot, list) else len(rot.keys()))
e0 = rot[0] if isinstance(rot, list) else next(iter(rot.values()))
print("keys:", e0.keys())
print("type field:", e0.get("type", None))
print("layer field:", e0.get("layer_idx", None), e0.get("layer", None), e0.get("idx", None))


<class 'list'> len: 48
keys: dict_keys(['type', 'layer_type', 'layer', 'rank', 'enabled', 'eigenvalues', 'eigenvectors', 'projector'])
type field: attention_projector
layer field: None 0 None


In [23]:
# =========================
# CELL 2 — Runner: builds paths to ROTATION MATRICES and evaluates all runs
# =========================
import os, json, logging
import torch

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

# -----------------------------
# Settings
# -----------------------------
MODEL_NAME = "Qwen/Qwen2-0.5B"
DEVICE = "cuda:0"
config.device = torch.device(DEVICE)

BASE_MODEL_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection"
BASE_SAVE_DIR  = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection"

CAL_DATASETS = ["wikitext2", "squad2"]
PROJECTION_MODES = ["attn_only", "mlp_only", "both"]
SPARSITY = 0.10

MODE_TO_SUFFIX = {
    "attn_only": "projection_attention_only",
    "mlp_only":  "projection_mlp_only",
    "both":      "projection_full",
}

TASKS = "squadv2"
BATCH_SIZE = 64
NUM_FEWSHOT = 0
LIMIT = 100  # e.g. 200 for quick test
DTYPE = torch.float16  # or torch.bfloat16

def run_tag(cal_dataset: str, mode: str) -> str:
    return f"{cal_dataset}_s{SPARSITY:.2f}_projOnly_{mode}".replace(".", "p")

def rot_path(cal_dataset: str, mode: str) -> str:
    tag = run_tag(cal_dataset, mode)
    model_dir = os.path.join(BASE_MODEL_DIR, tag)
    fname = f"Qwen2-0.5B_{SPARSITY}_{MODE_TO_SUFFIX[mode]}_rotation_matrices.pt"
    return os.path.join(model_dir, fname)

def run_eval_for_model(cal_dataset: str, mode: str):
    tag = run_tag(cal_dataset, mode)
    rpath = rot_path(cal_dataset, mode)
    save_dir = os.path.join(BASE_SAVE_DIR, tag)

    if not os.path.isfile(rpath):
        model_dir = os.path.dirname(rpath)
        raise FileNotFoundError(f"Rotation matrices not found: {rpath}\nDir listing: {os.listdir(model_dir)}")

    print("\n=====================================================")
    print(f"Evaluating PROJ-ONLY (hooks) | dataset={cal_dataset}, mode={mode}")
    print(f"Rot mats:  {rpath}")
    print(f"Save dir:  {save_dir}")
    print("=====================================================\n")

    args = {
        "model": MODEL_NAME,
        "rot_path": rpath,
        "mode": mode,
        "sparsity": SPARSITY,
        "batch_size": BATCH_SIZE,
        "tasks": TASKS,
        "num_fewshot": NUM_FEWSHOT,
        "limit": LIMIT,
        "save_dir": save_dir,
        "task_manager": None,
        "dtype": DTYPE,
    }
    return eval_projection_only_from_rot(args)

for ds in CAL_DATASETS:
    for mode in PROJECTION_MODES:
        run_eval_for_model(ds, mode)



Evaluating PROJ-ONLY (hooks) | dataset=wikitext2, mode=attn_only
Rot mats:  /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt
Save dir:  /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection/wikitext2_s0p10_projOnly_attn_only

INFO - Running Evaluation (projection-only via rotation matrices/hooks)
INFO - Base model: Qwen/Qwen2-0.5B
INFO - Rotation matrices: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt
INFO - Mode: attn_only | Sparsity tag: 0.10
INFO - Loading Qwen/Qwen2-0.5B config and model weights from Hugging Face


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO - Loading model done
[HOOKS] layer_path='model.layers' n_layers=24 | hooks=72 | matched_attn=24 | matched_mlp=0
hooks: 72
P diag mean: 0.8928571343421936 P diag std: 0.15071286261081696
P offdiag mean abs: 0.006684030871838331
INFO - Attached hooks: 72
WARNING - `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
WARNING - Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
INFO - Selected Tasks: ['squadv2']
INFO - Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
WARNING - Overwriting default num_fewshot of squadv2 from None to 0
INFO - Building contexts for None on rank 0...


100%|██████████| 100/100 [00:00<00:00, 122892.00it/s]

INFO - Running generate_until requests


...

INFO - Running generate_until requests



Running generate_until requests: 100%|██████████| 100/100 [03:00<00:00,  1.81s/it]

INFO - Running loglikelihood requests



Running loglikelihood requests: 100%|██████████| 100/100 [00:00<00:00, 267.36it/s]
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote

INFO - Saved results to /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection/squad2_s0p10_projOnly_both/results_projOnlyHooks_s0.10_both_squadv2.json


In [10]:
import os, json
import pandas as pd

BASE_EVAL_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection"

CAL_DATASETS = ["wikitext2", "squad2"]
PROJECTION_MODES = ["attn_only", "mlp_only", "both"]
SPARSITY = 0.10

def _run_tag(ds, mode, sparsity=SPARSITY):
    return f"{ds}_s{sparsity:.2f}_projOnly_{mode}".replace(".", "p")

rows = []
missing_files = []

for ds in CAL_DATASETS:
    for mode in PROJECTION_MODES:
        run_tag = _run_tag(ds, mode)
        run_dir = os.path.join(BASE_EVAL_DIR, run_tag)
        json_path = os.path.join(run_dir, f"results_projectionOnly_s{SPARSITY:.2f}_squadv2.json")

        if not os.path.isfile(json_path):
            missing_files.append(json_path)
            continue

        with open(json_path, "r") as f:
            data = json.load(f)

        # lm-eval stores squadv2 metrics under the "squadv2" key
        m = data.get("squadv2", {})

        # Common keys you’ll see (like in your screenshot):
        # "exact,none", "f1,none", "best_exact,none", "best_f1,none",
        # plus HasAns/NoAns splits.
        row = {
            "cal_dataset": ds,
            "mode": mode,
            "sparsity": SPARSITY,
            "exact": m.get("exact,none"),
            "f1": m.get("f1,none"),
            "best_exact": m.get("best_exact,none"),
            "best_f1": m.get("best_f1,none"),
            "HasAns_exact": m.get("HasAns_exact,none"),
            "HasAns_f1": m.get("HasAns_f1,none"),
            "NoAns_exact": m.get("NoAns_exact,none"),
            "NoAns_f1": m.get("NoAns_f1,none"),
            "file": json_path,
        }
        rows.append(row)

df = pd.DataFrame(rows)

# Nice ordering
if not df.empty:
    df = df.sort_values(["cal_dataset", "mode"]).reset_index(drop=True)

display(df)

# Optional: save a summary CSV next to the results
out_csv = os.path.join(BASE_EVAL_DIR, f"summary_projectionOnly_s{SPARSITY:.2f}_squadv2.csv")
df.to_csv(out_csv, index=False)
print(f"Saved summary CSV to: {out_csv}")

# Optional: show missing files (if any)
if missing_files:
    print("\nMissing result files:")
    for p in missing_files:
        print(" -", p)


,cal_dataset,mode,sparsity,exact,f1,best_exact,best_f1,HasAns_exact,HasAns_f1,NoAns_exact,NoAns_f1,file
0,squad2,attn_only,0.1,50.2,50.200000,50.2,50.2,0.0,0.000000,100.000000,100.000000,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
1,squad2,both,0.1,0.5,0.500000,50.2,50.2,0.0,0.000000,0.996016,0.996016,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
2,squad2,mlp_only,0.1,1.7,1.705405,50.2,50.2,0.0,0.010854,3.386454,3.386454,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
3,wikitext2,attn_only,0.1,50.2,50.200000,50.2,50.2,0.0,0.000000,100.000000,100.000000,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
4,wikitext2,both,0.1,0.2,0.206897,50.2,50.2,0.0,0.013848,0.398406,0.398406,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
5,wikitext2,mlp_only,0.1,1.3,1.300000,50.2,50.2,0.0,0.000000,2.589641,2.589641,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...


Saved summary CSV to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection/summary_projectionOnly_s0.10_squadv2.csv


In [24]:
import os, json
import pandas as pd

BASE_EVAL_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection"

CAL_DATASETS = ["wikitext2", "squad2"]
PROJECTION_MODES = ["attn_only", "mlp_only", "both"]
SPARSITY = 0.10
TASK = "squadv2"

def _run_tag(ds, mode, sparsity=SPARSITY):
    return f"{ds}_s{sparsity:.2f}_projOnly_{mode}".replace(".", "p")

rows = []
missing_files = []

for ds in CAL_DATASETS:
    for mode in PROJECTION_MODES:
        run_tag = _run_tag(ds, mode)
        run_dir = os.path.join(BASE_EVAL_DIR, run_tag)

        # NEW filename template:
        # results_projOnlyHooks_s0.10_mlp_only_squadv2.json
        json_name = f"results_projOnlyHooks_s{SPARSITY:.2f}_{mode}_{TASK}.json"
        json_path = os.path.join(run_dir, json_name)

        if not os.path.isfile(json_path):
            missing_files.append(json_path)
            continue

        with open(json_path, "r") as f:
            data = json.load(f)

        # Your eval code writes lm-eval's ["results"] dict.
        # For squadv2, metrics are usually under key "squadv2".
        m = data.get(TASK, {})

        row = {
            "cal_dataset": ds,
            "mode": mode,
            "sparsity": SPARSITY,
            "exact": m.get("exact,none"),
            "f1": m.get("f1,none"),
            "best_exact": m.get("best_exact,none"),
            "best_f1": m.get("best_f1,none"),
            "HasAns_exact": m.get("HasAns_exact,none"),
            "HasAns_f1": m.get("HasAns_f1,none"),
            "NoAns_exact": m.get("NoAns_exact,none"),
            "NoAns_f1": m.get("NoAns_f1,none"),
            "file": json_path,
        }
        rows.append(row)

df = pd.DataFrame(rows)

if not df.empty:
    df = df.sort_values(["cal_dataset", "mode"]).reset_index(drop=True)

display(df)

out_csv = os.path.join(BASE_EVAL_DIR, f"summary_projOnlyHooks_s{SPARSITY:.2f}_{TASK}.csv")
df.to_csv(out_csv, index=False)
print(f"Saved summary CSV to: {out_csv}")

if missing_files:
    print("\nMissing result files:")
    for p in missing_files:
        print(" -", p)


,cal_dataset,mode,sparsity,exact,f1,best_exact,best_f1,HasAns_exact,HasAns_f1,NoAns_exact,NoAns_f1,file
0,squad2,attn_only,0.1,0.0,3.450603,55.0,55.000000,0.000000,7.668008,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
1,squad2,both,0.1,0.0,6.950487,55.0,55.142857,0.000000,15.445527,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
2,squad2,mlp_only,0.1,5.0,11.425752,55.0,55.000000,11.111111,25.390561,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
3,wikitext2,attn_only,0.1,1.0,5.990645,55.0,55.000000,2.222222,13.312545,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
4,wikitext2,both,0.1,0.0,0.041464,55.0,55.001726,0.000000,0.092142,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...
5,wikitext2,mlp_only,0.1,1.0,1.807410,55.0,55.000000,2.222222,4.016466,0.0,0.0,/content/drive/MyDrive/TUM/Pratikum/SliceGPTEx...


Saved summary CSV to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval_projection/summary_projOnlyHooks_s0.10_squadv2.csv


In [4]:
# ============================================
# CELL 1 — Imports + prompt + generation helper
# ============================================
import os
import torch
from datasets import load_dataset

from slicegpt import hf_utils
from slicegpt.config import config

def make_prompt_squadv2(context: str, question: str) -> str:
    return (
        "Answer the question using ONLY the context. "
        "If the answer is not in the context, output: unanswerable.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

@torch.inference_mode()
def generate_answer(model, tokenizer, prompt: str, max_new_tokens: int = 48, max_length: int = 2048) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)

    ans = decoded.split("Answer:", 1)[-1].strip()
    ans = ans.split("\n")[0].strip()
    return ans


In [5]:
# ============================================
# CELL 2 — Config + RUNS pointing to ROTATION MATRICES
# ============================================
MODEL_NAME = "Qwen/Qwen2-0.5B"
DEVICE = "cuda:0"
config.device = torch.device(DEVICE)

BASE_MODEL_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection"
SPARSITY = 0.10

CAL_DATASETS = ["wikitext2", "squad2"]
MODES = ["attn_only", "mlp_only", "both"]

MODE_TO_SUFFIX = {
    "attn_only": "projection_attention_only",
    "mlp_only":  "projection_mlp_only",
    "both":      "projection_full",
}

def run_tag(ds: str, mode: str) -> str:
    return f"{ds}_s{SPARSITY:.2f}_projOnly_{mode}".replace(".", "p")

def rot_path(ds: str, mode: str) -> str:
    tag = run_tag(ds, mode)
    fname = f"Qwen2-0.5B_{SPARSITY}_{MODE_TO_SUFFIX[mode]}_rotation_matrices.pt"
    return os.path.join(BASE_MODEL_DIR, tag, fname)

RUNS = []
for ds in CAL_DATASETS:
    for mode in MODES:
        RUNS.append({
            "dataset": ds,
            "mode": mode,
            "tag": run_tag(ds, mode),
            "rot": rot_path(ds, mode),
        })

ds_val = load_dataset("squad_v2", split="validation")
N = 5
examples = [ds_val[i] for i in range(N)]

print("Will debug these rotation matrices:")
for r in RUNS:
    print(" -", r["rot"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Will debug these rotation matrices:
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_mlp_only/Qwen2-0.5B_0.1_projection_mlp_only_rotation_matrices.pt
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_both/Qwen2-0.5B_0.1_projection_full_rotation_matrices.pt
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/squad2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/squad2_s0p10_projOnly_mlp_only/Qwen2-0.5B_0.1_projection_mlp_only_rotation_matrices.pt
 - /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/squad2_s0p10_projOnly_both/Qwen2-0.5B_0.1_projection

In [6]:
# ============================================
# CELL 3 — Load base model + attach projection hooks from rotation matrices
# Uses your real schema:
#   entry["type"] = "attention_projector" / "mlp_projector"
#   entry["layer"] = layer idx
# and robustly finds HF layer blocks under model.model.layers etc.
# ============================================
import torch
import os

def _as_tensor(x):
    return x if isinstance(x, torch.Tensor) else torch.tensor(x)

def _iter_entries(rotation_matrices):
    if isinstance(rotation_matrices, list):
        return rotation_matrices
    if isinstance(rotation_matrices, dict):
        return rotation_matrices.values()
    raise TypeError(f"rotation_matrices must be list or dict, got {type(rotation_matrices)}")

def _get_projector(entry, device, dtype):
    P = entry.get("projector", None)
    if P is None:
        Q = entry.get("eigenvectors", None)
        if Q is None:
            raise ValueError(f"Entry missing both 'projector' and 'eigenvectors'. Keys={list(entry.keys())}")
        Q = _as_tensor(Q)
        P = Q @ Q.T
    else:
        P = _as_tensor(P)
    return P.to(device=device, dtype=dtype)

def _get_hf_layers(causal_lm_model):
    """
    Return the list-like container of transformer blocks for HF models.
    Works for Qwen2ForCausalLM: model.model.layers
    """
    candidate_paths = [
        "model.layers",            # Qwen2/LLaMA-style under ForCausalLM wrapper
        "layers",                  # sometimes directly
        "transformer.h",           # GPT-style
        "model.decoder.layers",    # some decoders
        "decoder.layers",
    ]

    for path in candidate_paths:
        obj = causal_lm_model
        ok = True
        for attr in path.split("."):
            if not hasattr(obj, attr):
                ok = False
                break
            obj = getattr(obj, attr)
        if ok:
            # ensure list-like
            _ = len(obj)
            return obj, path

    raise AttributeError(
        "Could not locate transformer layers. Tried: "
        + ", ".join(candidate_paths)
    )

def apply_projection_hooks_qwen2(model_adapter, rotation_matrices, mode: str):
    """
    Hook implementation that does NOT rely on adapter methods.
    It hooks HF modules directly:
      - Attention: self_attn.q_proj/k_proj/v_proj
      - MLP:      mlp.up_proj/gate_proj
    """
    model = model_adapter.model  # e.g., Qwen2ForCausalLM
    device = model.device
    dtype  = next(model.parameters()).dtype

    layers, layer_path = _get_hf_layers(model)

    hooks = []
    matched_attn = 0
    matched_mlp  = 0

    entries = list(_iter_entries(rotation_matrices))
    print(f"  [INFO] rot type={type(rotation_matrices).__name__}, entries={len(entries)}, layer_path='{layer_path}', n_layers={len(layers)}")

    for entry in entries:
        if not isinstance(entry, dict):
            continue
        if not entry.get("enabled", True):
            continue

        # your schema uses "layer"
        layer = int(entry["layer"])
        if layer < 0 or layer >= len(layers):
            continue

        t = str(entry.get("type", "")).lower()
        is_attn = "attention" in t
        is_mlp  = "mlp" in t

        # only load P if we will use it
        if (is_attn and mode in ("attn_only", "both")) or (is_mlp and mode in ("mlp_only", "both")):
            P = _get_projector(entry, device=device, dtype=dtype)

            def pre_hook(module, inputs, P=P):
                x = inputs[0]
                return (x @ P.T,) + inputs[1:]

            block = layers[layer]

            if is_attn and mode in ("attn_only", "both"):
                attn = getattr(block, "self_attn", None) or getattr(block, "attention", None)
                if attn is None:
                    raise AttributeError(f"Layer {layer} has no self_attn/attention module")

                q = getattr(attn, "q_proj", None)
                k = getattr(attn, "k_proj", None)
                v = getattr(attn, "v_proj", None)
                if q is None or k is None or v is None:
                    raise AttributeError(f"Layer {layer} attention missing q/k/v proj. attn attrs={dir(attn)}")

                hooks += [
                    q.register_forward_pre_hook(pre_hook),
                    k.register_forward_pre_hook(pre_hook),
                    v.register_forward_pre_hook(pre_hook),
                ]
                matched_attn += 1

            if is_mlp and mode in ("mlp_only", "both"):
                mlp = getattr(block, "mlp", None) or getattr(block, "feed_forward", None)
                if mlp is None:
                    raise AttributeError(f"Layer {layer} has no mlp/feed_forward module")

                up   = getattr(mlp, "up_proj", None)
                gate = getattr(mlp, "gate_proj", None)
                if up is None or gate is None:
                    raise AttributeError(f"Layer {layer} MLP missing up/gate proj. mlp attrs={dir(mlp)}")

                hooks += [
                    up.register_forward_pre_hook(pre_hook),
                    gate.register_forward_pre_hook(pre_hook),
                ]
                matched_mlp += 1

    print(f"  [HOOKS] hooks={len(hooks)} | matched_attn_entries={matched_attn} | matched_mlp_entries={matched_mlp}")
    return hooks

loaded = {}

for r in RUNS:
    rot = r["rot"]
    assert os.path.exists(rot), f"Missing rotation matrices file: {rot}"

    print(f"\n[LOAD] {r['dataset']} | {r['mode']}")
    print(" rot:", rot)

    model_adapter, tokenizer = hf_utils.get_model_and_tokenizer(
        MODEL_NAME,
        model_path=None,
        token=None,
        dtype=torch.float16,  # or torch.bfloat16
    )

    m = model_adapter.model.to(config.device)
    m.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    rotation_matrices = torch.load(rot, map_location="cpu")
    hooks = apply_projection_hooks_qwen2(model_adapter, rotation_matrices, mode=r["mode"])

    key = f"{r['dataset']}_{r['mode']}"
    loaded[key] = (m, tokenizer, hooks, rot)  # keep hooks alive + keep path
    print(f"  [OK] loaded {key} | hooks attached: {len(hooks)}")



[LOAD] wikitext2 | attn_only
 rot: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/wikitext2_s0p10_projOnly_attn_only/Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  [INFO] rot type=list, entries=48, layer_path='model.layers', n_layers=24
  [HOOKS] hooks=72 | matched_attn_entries=24 | matched_mlp_entries=0
  [OK] loaded squad2_attn_only | hooks attached: 72

[LOAD] squad2 | mlp_only
 rot: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/squad2_s0p10_projOnly_mlp_only/Qwen2-0.5B_0.1_projection_mlp_only_rotation_matrices.pt


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  [INFO] rot type=list, entries=48, layer_path='model.layers', n_layers=24
  [HOOKS] hooks=48 | matched_attn_entries=0 | matched_mlp_entries=24
  [OK] loaded squad2_mlp_only | hooks attached: 48

[LOAD] squad2 | both
 rot: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_projection/squad2_s0p10_projOnly_both/Qwen2-0.5B_0.1_projection_full_rotation_matrices.pt


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  [INFO] rot type=list, entries=48, layer_path='model.layers', n_layers=24
  [HOOKS] hooks=120 | matched_attn_entries=24 | matched_mlp_entries=24
  [OK] loaded squad2_both | hooks attached: 120


In [7]:
# ============================================
# CELL 4 — Run debug generation on the loaded variants
# Adds a tiny sanity print so you KNOW the models differ:
#   shows #hooks and a hash-like hint (rot filename)
# ============================================
for i, ex in enumerate(examples):
    context, question = ex["context"], ex["question"]
    golds = ex["answers"]["text"]
    gold = golds[0] if len(golds) > 0 else "unanswerable"

    prompt = make_prompt_squadv2(context, question)

    print("\n" + "=" * 120)
    print(f"[{i}] Q: {question}")
    print(f"Gold: {gold}")

    for key, (m, tok, hooks, rot_path_used) in loaded.items():
        ans = generate_answer(m, tok, prompt, max_new_tokens=48)
        print(f"\n--- {key} | hooks={len(hooks)} | rot={os.path.basename(rot_path_used)} ---")
        print(ans)



[0] Q: In what country is Normandy located?
Gold: France

--- wikitext2_attn_only | hooks=72 | rot=Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt ---
Normandy is located in France.

--- wikitext2_mlp_only | hooks=48 | rot=Qwen2-0.5B_0.1_projection_mlp_only_rotation_matrices.pt ---
The answer is not in the context of the question.

--- wikitext2_both | hooks=120 | rot=Qwen2-0.5B_0.1_projection_full_rotation_matrices.pt ---
In what country is Normandy located? answer: In what country is Normandy located? answer: In what country is Normandy located? answer: In what country is Normandy located? answer: In what country is Normandy located?

--- squad2_attn_only | hooks=72 | rot=Qwen2-0.5B_0.1_projection_attention_only_rotation_matrices.pt ---
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in

--- squad2_mlp_only | hooks=48 | rot=Qwen2-0.5B_0.1_projection_mlp_only_rota